In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import resample_poly, butter, filtfilt, iirnotch, welch
from sklearn.preprocessing import MinMaxScaler
import joblib


In [ ]:
# load txt openbci
data = pd.read_csv("../Pattern Recognition.txt", delimiter=", ")  # sesuaikan delimiter
eeg_columns = [col for col in data.columns if "EXG Channel" in col]

eeg = data[eeg_columns].values
fs_original = 125

print("Shape EEG:", eeg.shape)
print("Channel used:", eeg_columns) # misalnya kolom pertama timestamp
fs_original = 125


In [ ]:
fs_target = 512

# rasio dalam bentuk integer
up = 512
down = 125

eeg_upsampled = resample_poly(eeg, up, down, axis=0)


In [ ]:
def notch_filter(signal, fs, freq=50, Q=30):
    b, a = iirnotch(freq, Q, fs)
    return filtfilt(b, a, signal, axis=0)

eeg_notch = notch_filter(eeg_upsampled, fs_target)


In [ ]:
def bandpass_filter(signal, fs, low=1, high=40, order=4):
    nyq = 0.5 * fs
    low = low / nyq
    high = high / nyq
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, signal, axis=0)

eeg_filtered = bandpass_filter(eeg_notch, fs_target)


In [ ]:
window_size = 2  # seconds
overlap = 0.5

samples_per_window = int(window_size * fs_target)
step = int(samples_per_window * (1 - overlap))

windows = []
for start in range(0, len(eeg_filtered) - samples_per_window, step):
    end = start + samples_per_window
    windows.append(eeg_filtered[start:end])

windows = np.array(windows)


In [ ]:
def extract_psd(window, fs):
    freqs, psd = welch(window, fs=fs, nperseg=fs*2, axis=0)
    
    # ambil hanya 0–40 Hz
    mask = freqs <= 40
    psd = psd[mask]
    
    # flatten channel
    return psd.flatten()

features = np.array([extract_psd(w, fs_target) for w in windows])


In [ ]:
scaler = MinMaxScaler()
features_scaled = scaler.fit_transform(features)

joblib.dump(scaler, "scaler_training.pkl")


In [ ]:
print("FSD:", features)

In [ ]:
data = pd.read_csv("../eeg-analysis\cognitive_state_discrimination_dataset - cognitive_state_discrimination_dataset.csv", delimiter=",")

In [ ]:
print(data[0:5])